# Notebook 07 - Combined Feature Model

Goal:

Test whether combining character n-gram features with handcrafted lexical features improves domain classification.

Previous best useful model from Notebook 06:

Character N-Gram + Logistic Regression

- Accuracy: ~0.881
- Macro F1: ~0.38
- Malware recall: ~0.54
- Phishing recall: ~0.48
- Spam recall: ~0.34

This notebook tests:

domain text -> character n-grams  
numeric lexical features -> scaled  
TLD -> one-hot encoded  
classifier -> Logistic Regression / SGD

Primary goal:

Improve minority-class detection, especially malware, phishing, and spam.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score
)

In [ ]:
DATA_PATH = "../data/processed/auspex_features_v1.csv"

features = pd.read_csv(DATA_PATH)

print(features.shape)
features.head()

In [ ]:
expected_columns = [
    "domain",
    "label",
    "length",
    "digit_count",
    "digit_ratio",
    "dot_count",
    "tld",
    "starts_with_digit",
    "contains_www",
    "entropy",
    "hyphen_count",
]

missing_columns = set(expected_columns) - set(features.columns)
extra_columns = set(features.columns) - set(expected_columns)

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

print("\nNull counts:")
print(features[expected_columns].isna().sum())

print("\nClass counts:")
print(features["label"].value_counts())

print("\nDuplicate domains:")
print(features["domain"].duplicated().sum())

assert len(missing_columns) == 0, "Missing expected columns."
assert "label" in features.columns, "Missing label column."
assert "domain" in features.columns, "Missing domain column."

In [ ]:
features = features.copy()

features["domain"] = features["domain"].astype(str).str.lower().str.strip()
features["label"] = features["label"].astype(str).str.lower().str.strip()
features["tld"] = features["tld"].fillna("unknown").astype(str).str.lower().str.strip()

numeric_features = [
    "length",
    "digit_count",
    "digit_ratio",
    "dot_count",
    "starts_with_digit",
    "contains_www",
    "entropy",
    "hyphen_count",
]

for col in numeric_features:
    features[col] = pd.to_numeric(features[col], errors="coerce")

features[numeric_features] = features[numeric_features].fillna(0)

print(features.dtypes)
print("\nRemaining nulls:")
print(features.isna().sum())

In [ ]:
X = features.drop(columns=["label"])
y = features["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

In [ ]:
LABEL_ORDER = ["benign", "malware", "phishing", "spam"]

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)

    print(model_name)
    print("-" * len(model_name))

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=LABEL_ORDER,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)

    print("\nConfusion Matrix:")
    print(cm)

    threat_true = y_test != "benign"
    threat_pred = y_pred != "benign"

    malicious_recall = recall_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    malicious_precision = precision_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    malicious_f1 = f1_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    threats_predicted_benign = ((y_test != "benign") & (y_pred == "benign")).sum()
    total_threats = (y_test != "benign").sum()

    print("\nBinary Threat Detection View:")
    print("Malicious recall:", malicious_recall)
    print("Malicious precision:", malicious_precision)
    print("Malicious F1:", malicious_f1)
    print("Threats predicted benign:", threats_predicted_benign, "/", total_threats)

    return {
        "model_name": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "malicious_recall": malicious_recall,
        "malicious_precision": malicious_precision,
        "malicious_f1": malicious_f1,
        "threats_predicted_benign": int(threats_predicted_benign),
        "total_threats": int(total_threats),
        "confusion_matrix": cm.tolist()
    }

In [ ]:
notebook_06_benchmark = {
    "model_name": "Notebook 06 - Pure N-Gram Logistic Regression",
    "accuracy": 0.8810734573564721,
    "macro_f1": 0.38,
    "malware_recall": 0.54,
    "phishing_recall": 0.48,
    "spam_recall": 0.34,
}

notebook_06_benchmark

In [ ]:
combined_preprocessor = ColumnTransformer(
    transformers=[
        (
            "domain_ngrams",
            HashingVectorizer(
                analyzer="char",
                ngram_range=(2, 5),
                n_features=2**18,
                alternate_sign=False
            ),
            "domain"
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "tld",
            OneHotEncoder(handle_unknown="ignore"),
            ["tld"]
        ),
    ],
    remainder="drop"
)

In [ ]:
sample_X = X_train.head(1000)

smoke_preprocessor = clone(combined_preprocessor)
sample_transformed = smoke_preprocessor.fit_transform(sample_X)

print("Sample transformed shape:", sample_transformed.shape)
print("Sample transformed type:", type(sample_transformed))

if hasattr(sample_transformed, "nnz"):
    density = sample_transformed.nnz / (sample_transformed.shape[0] * sample_transformed.shape[1])
    print("Sample sparse density:", density)

In [ ]:
combined_sgd = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("classifier", SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        max_iter=30,
        random_state=42,
        n_jobs=-1
    ))
])

combined_sgd.fit(X_train, y_train)

print("Combined SGD training complete.")

In [ ]:
combined_sgd_results = evaluate_model(
    combined_sgd,
    X_test,
    y_test,
    "Combined Character N-Grams + Lexical Features + SGD"
)

combined_sgd_results

In [ ]:
combined_logreg = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
])

combined_logreg.fit(X_train, y_train)

print("Combined Logistic Regression training complete.")

In [ ]:
combined_logreg_results = evaluate_model(
    combined_logreg,
    X_test,
    y_test,
    "Combined Character N-Grams + Lexical Features + Logistic Regression"
)

combined_logreg_results